# Combined coverage experiment: full cohort + age 65--70

This notebook runs the revised repeated-labeling experiments using:

- subject-level out-of-fold autorater probabilities
- scan-level age matched by `subject_id + Acq_Date`
- CRE, labeled-only Bayes, binary difference estimator, continuous-score PPI, and PPI++

Outputs:

- `coverage_all_subject_oof_scan_age.csv`
- `coverage_6570_subject_oof_scan_age.csv`
- `coverage_all_6570_subject_oof_scan_age_combined.csv`
- coverage and interval-width figures for both datasets

In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [ ]:
# Run this once if needed.
# If packages are already installed, this cell is harmless.
%pip install -q pandas numpy scipy scikit-learn tqdm matplotlib

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as student_t
from tqdm import tqdm

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

In [ ]:
# ============================================================
# 0) Paths and global settings
# ============================================================

FULL_CSV = REPO_ROOT / "results/autorater/autorater_predictions_all_subject_oof_with_scan_age.csv"
AGE6570_CSV = REPO_ROOT / "results/autorater/autorater_predictions_6570_subject_oof_scan_age.csv"

OUTPUT_DIR = REPO_ROOT / "results/coverage"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FULL_CSV = OUTPUT_DIR / "coverage_all_subject_oof_scan_age.csv"
OUTPUT_6570_CSV = OUTPUT_DIR / "coverage_6570_subject_oof_scan_age.csv"
OUTPUT_COMBINED_CSV = OUTPUT_DIR / "coverage_all_6570_subject_oof_scan_age_combined.csv"

THRESHOLD_POLICY = "fixed_0.5"
THRESHOLD = 0.5

# For a quick test, set NSIM = 50.
# For final revision results, use NSIM = 500.
NSIM = 500

LABEL_SIZES = [10, 20, 40, 80]

PRIORS = [
    {"name": "uniform", "alpha": 1.0, "beta": 1.0},
    {"name": "jeffreys", "alpha": 0.5, "beta": 0.5},
]

POSTERIOR_DRAWS = 5000
BOOTSTRAP_B = 1000
SEED = 2025
Z975 = 1.959963984540054

SAVE_FIGURES = True

DISPLAY_NAME = {
    "full": "Full cohort",
    "age_65_70": "Age 65--70 subset",
}

In [ ]:
# ============================================================
# 1) Loading and QC utilities
# ============================================================

def load_prediction_csv(csv_path: Path, threshold: float = 0.5) -> pd.DataFrame:
    """
    Load subject-level OOF prediction table.

    Required columns:
      subject_id, H, autorater_prediction

    Optional but expected for this revision:
      Age, oof_fold, Acq_Date
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find input CSV: {csv_path}")

    df = pd.read_csv(csv_path).copy()
    df.columns = [c.strip() for c in df.columns]

    required = {"subject_id", "H", "autorater_prediction"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path.name} missing required columns: {missing}")

    df["subject_id"] = df["subject_id"].astype(str).str.strip()
    df["H"] = pd.to_numeric(df["H"], errors="coerce").astype(int)
    df["A_prob"] = pd.to_numeric(df["autorater_prediction"], errors="coerce")

    df = df.dropna(subset=["A_prob", "H", "subject_id"]).copy()
    df = df[df["A_prob"].between(0, 1, inclusive="both")].copy()

    df["A_class"] = (df["A_prob"] >= threshold).astype(int)

    if "Age" in df.columns:
        df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

    return df.reset_index(drop=True)


def qc_dataset(df: pd.DataFrame, name: str, threshold: float = 0.5) -> None:
    """
    Print dataset QC information for the revision audit trail.
    """
    print("=" * 90)
    print(f"QC: {name}")
    print("=" * 90)

    print("Rows:", len(df))
    print("Unique subjects:", df["subject_id"].nunique())
    print("True prevalence g:", float(df["H"].mean()))
    print(f"P(A=1) @ threshold {threshold}:", float(df["A_class"].mean()))

    if "Age" in df.columns:
        print("Age missing:", int(df["Age"].isna().sum()))
        print("Age range:", float(df["Age"].min()), "to", float(df["Age"].max()))

    if "oof_fold" in df.columns:
        per_subject_fold_n = df.groupby("subject_id")["oof_fold"].nunique()
        print("Subjects spanning multiple OOF folds:", int((per_subject_fold_n > 1).sum()))
        print("Max folds per subject:", int(per_subject_fold_n.max()))
        print("OOF fold row counts:")
        print(df["oof_fold"].value_counts().sort_index())

    y = df["H"].to_numpy(dtype=int)
    p = df["A_prob"].to_numpy(dtype=float)
    a = df["A_class"].to_numpy(dtype=int)

    print("Autorater metrics")
    print("AUC:", float(roc_auc_score(y, p)))
    print("ACC @ threshold:", float(accuracy_score(y, a)))

    tn, fp, fn, tp = confusion_matrix(y, a, labels=[0, 1]).ravel()
    print("TN, FP, FN, TP:", int(tn), int(fp), int(fn), int(tp))
    print("TPR:", float(tp / (tp + fn)) if (tp + fn) else np.nan)
    print("TNR:", float(tn / (tn + fp)) if (tn + fp) else np.nan)

In [ ]:
# ============================================================
# 2) Load full cohort and age 65--70 subset
# ============================================================

full_df = load_prediction_csv(FULL_CSV, threshold=THRESHOLD)
age6570_df = load_prediction_csv(AGE6570_CSV, threshold=THRESHOLD)

qc_dataset(full_df, "Full cohort", threshold=THRESHOLD)
qc_dataset(age6570_df, "Age 65--70", threshold=THRESHOLD)

In [ ]:
# ============================================================
# 3) Estimators
# ============================================================

def posterior_summary(draws: np.ndarray):
    draws = np.asarray(draws, dtype=float)
    return float(draws.mean()), (
        float(np.quantile(draws, 0.025)),
        float(np.quantile(draws, 0.975))
    )


def cre_estimator(
    df: pd.DataFrame,
    labeled_idx: np.ndarray,
    alpha: float,
    beta: float,
    size: int,
    rng: np.random.Generator,
):
    """
    Conjugate Bayesian chain-rule estimator.

    Uses:
      full pool: thresholded A_class
      labeled subset: A_class and H

    g = P(A=1) P(H=1 | A=1) + P(A=0) P(H=1 | A=0)
    """
    A_all = df["A_class"].to_numpy(dtype=int)
    N_all = len(A_all)
    nA1_all = int(A_all.sum())

    sub = df.iloc[labeled_idx]
    A_lab = sub["A_class"].to_numpy(dtype=int)
    H_lab = sub["H"].to_numpy(dtype=int)

    mask1 = A_lab == 1
    mask0 = A_lab == 0

    n1 = int(mask1.sum())
    n0 = int(mask0.sum())

    H1 = int(H_lab[mask1].sum())
    H0 = int(H_lab[mask0].sum())

    thetaA = rng.beta(alpha + nA1_all, beta + N_all - nA1_all, size=size)
    thetaH1 = rng.beta(alpha + H1, beta + n1 - H1, size=size)
    thetaH0 = rng.beta(alpha + H0, beta + n0 - H0, size=size)

    g_draws = thetaA * thetaH1 + (1.0 - thetaA) * thetaH0
    return posterior_summary(g_draws)


def labeled_only_estimator(
    df: pd.DataFrame,
    labeled_idx: np.ndarray,
    alpha: float,
    beta: float,
    size: int,
    rng: np.random.Generator,
):
    """
    Labeled-only Bayesian baseline.
    Labeled-only Beta-Bernoulli posterior summary.
    """
    sub = df.iloc[labeled_idx]
    H = sub["H"].to_numpy(dtype=int)

    n = len(H)
    H_sum = int(H.sum())

    draws = rng.beta(alpha + H_sum, beta + n - H_sum, size=size)
    return posterior_summary(draws)


def binary_difference_estimator(
    df: pd.DataFrame,
    labeled_idx: np.ndarray,
    bootstrap_B: int,
    rng: np.random.Generator,
):
    """
    Binary difference estimator using thresholded A_class.

    g_hat = mean(A_class full pool) + mean(H - A_class labeled subset)
    """
    A_bar = float(df["A_class"].mean())

    sub = df.iloc[labeled_idx]
    resid = (
        sub["H"].to_numpy(dtype=float)
        - sub["A_class"].to_numpy(dtype=float)
    )

    g_hat = A_bar + float(resid.mean())

    if len(resid) <= 1:
        return float(g_hat), (np.nan, np.nan)

    boots = np.empty(bootstrap_B, dtype=float)
    for b in range(bootstrap_B):
        rb = rng.choice(resid, size=len(resid), replace=True)
        boots[b] = A_bar + float(rb.mean())

    ci = np.quantile(boots, [0.025, 0.975])
    return float(g_hat), (float(ci[0]), float(ci[1]))


def ppi_analytic_estimator(df: pd.DataFrame, labeled_idx: np.ndarray):
    """
    Prior-free continuous-score PPI.

    Uses:
      full pool: A_prob
      labeled subset: H - A_prob

    This is intentionally distinct from the binary difference estimator.
    """
    p_all = df["A_prob"].to_numpy(dtype=float)
    N_all = len(p_all)

    p_bar = float(p_all.mean())
    var_p = float(np.var(p_all, ddof=1)) if N_all > 1 else 0.0

    sub = df.iloc[labeled_idx]
    H_lab = sub["H"].to_numpy(dtype=float)
    p_lab = sub["A_prob"].to_numpy(dtype=float)

    resid = H_lab - p_lab
    n_lab = len(resid)

    r_bar = float(resid.mean())
    var_r = float(np.var(resid, ddof=1)) if n_lab > 1 else 0.0

    g_hat = p_bar + r_bar
    se = np.sqrt(var_p / N_all + var_r / n_lab)

    crit = Z975 if n_lab >= 30 else float(student_t.ppf(0.975, df=max(n_lab - 1, 1)))
    return float(g_hat), (float(g_hat - crit * se), float(g_hat + crit * se))


def ppi_plus_estimator(df: pd.DataFrame, labeled_idx: np.ndarray):
    """
    Simple PPI++-style power-tuned continuous-score baseline.

    g_lambda = lambda * mean(p_all) + mean(H_labeled - lambda * p_labeled)

    lambda is chosen to approximately minimize:
      lambda^2 Var(p_all)/N + Var(H - lambda p_labeled)/n

    We clip lambda to [0, 1] for a conservative deployment comparison.
    """
    p_all = df["A_prob"].to_numpy(dtype=float)
    N_all = len(p_all)

    sub = df.iloc[labeled_idx]
    H_lab = sub["H"].to_numpy(dtype=float)
    p_lab = sub["A_prob"].to_numpy(dtype=float)
    n_lab = len(H_lab)

    if n_lab < 2 or N_all < 2:
        g_hat, ci = ppi_analytic_estimator(df, labeled_idx)
        return g_hat, ci, 1.0

    var_p_all = float(np.var(p_all, ddof=1))
    var_p_lab = float(np.var(p_lab, ddof=1))
    cov_hp = float(np.cov(H_lab, p_lab, ddof=1)[0, 1])

    denom = (var_p_lab / n_lab) + (var_p_all / N_all)
    lam = 0.0 if denom <= 0 or not np.isfinite(denom) else (cov_hp / n_lab) / denom
    lam = float(np.clip(lam, 0.0, 1.0))

    resid = H_lab - lam * p_lab
    g_hat = lam * float(p_all.mean()) + float(resid.mean())

    var_resid = float(np.var(resid, ddof=1)) if n_lab > 1 else 0.0
    se = np.sqrt((lam ** 2) * var_p_all / N_all + var_resid / n_lab)

    crit = Z975 if n_lab >= 30 else float(student_t.ppf(0.975, df=max(n_lab - 1, 1)))
    return float(g_hat), (float(g_hat - crit * se), float(g_hat + crit * se)), lam


def coverage_and_width(ci, truth: float):
    lo, hi = ci

    if np.isnan(lo) or np.isnan(hi):
        return False, np.nan

    covered = (lo <= truth <= hi)
    width = hi - lo

    return bool(covered), float(width)

In [ ]:
# ============================================================
# 4) Repeated-labeling experiment
# ============================================================

def run_coverage_experiment(
    df: pd.DataFrame,
    dataset_name: str,
    input_csv_path: Path,
    output_csv: Path,
    seed_offset: int = 0,
) -> pd.DataFrame:
    """
    Run repeated-labeling experiment for one dataset.

    Important reproducibility choices:
      - same labeled index sets are reused for Uniform and Jeffreys priors
      - PPI, PPI++, and binary Diff are prior-free and therefore repeat across prior rows
      - input_csv_path and output_csv are recorded separately
    """
    print("#" * 100)
    print(f"Running coverage experiment: {dataset_name}")
    print("#" * 100)

    N = len(df)
    g_true = float(df["H"].mean())

    print("Rows:", N)
    print("Subjects:", df["subject_id"].nunique())
    print("g_true:", g_true)

    rng_master = np.random.default_rng(SEED + seed_offset)

    label_index_sets = {
        n: [rng_master.choice(N, size=n, replace=False) for _ in range(NSIM)]
        for n in LABEL_SIZES
    }

    all_rows = []

    for prior in PRIORS:
        pname = prior["name"]
        alpha = prior["alpha"]
        beta = prior["beta"]

        for n_labels in LABEL_SIZES:
            cov_counts = {
                "chain": 0,
                "labeled_only": 0,
                "diff": 0,
                "ppi": 0,
                "ppipp": 0,
            }

            widths = {
                "chain": [],
                "labeled_only": [],
                "diff": [],
                "ppi": [],
                "ppipp": [],
            }

            ppipp_lambdas = []

            desc = f"{dataset_name} | {pname} | labels={n_labels}"

            for rep, labeled_idx in enumerate(tqdm(label_index_sets[n_labels], desc=desc)):
                rep_seed = SEED + seed_offset + 100000 * rep + 1000 * n_labels
                rep_rng = np.random.default_rng(rep_seed)
                post_rng = np.random.default_rng(rep_seed + 17)

                # CRE
                _, ci_c = cre_estimator(
                    df,
                    labeled_idx,
                    alpha=alpha,
                    beta=beta,
                    size=POSTERIOR_DRAWS,
                    rng=post_rng,
                )
                ok, w = coverage_and_width(ci_c, g_true)
                cov_counts["chain"] += int(ok)
                widths["chain"].append(w)

                # Labeled-only Bayes
                _, ci_l = labeled_only_estimator(
                    df,
                    labeled_idx,
                    alpha=alpha,
                    beta=beta,
                    size=POSTERIOR_DRAWS,
                    rng=post_rng,
                )
                ok, w = coverage_and_width(ci_l, g_true)
                cov_counts["labeled_only"] += int(ok)
                widths["labeled_only"].append(w)

                # Binary difference estimator
                _, ci_d = binary_difference_estimator(
                    df,
                    labeled_idx,
                    bootstrap_B=BOOTSTRAP_B,
                    rng=rep_rng,
                )
                ok, w = coverage_and_width(ci_d, g_true)
                cov_counts["diff"] += int(ok)
                widths["diff"].append(w)

                # Continuous-score PPI
                _, ci_p = ppi_analytic_estimator(df, labeled_idx)
                ok, w = coverage_and_width(ci_p, g_true)
                cov_counts["ppi"] += int(ok)
                widths["ppi"].append(w)

                # PPI++
                _, ci_pp, lam = ppi_plus_estimator(df, labeled_idx)
                ok, w = coverage_and_width(ci_pp, g_true)
                cov_counts["ppipp"] += int(ok)
                widths["ppipp"].append(w)
                ppipp_lambdas.append(lam)

            row = {
                "dataset": dataset_name,
                "input_csv_path": str(input_csv_path),
                "output_csv": str(output_csv),
                "threshold_policy": THRESHOLD_POLICY,
                "threshold": THRESHOLD,
                "n_rows": len(df),
                "n_subjects": df["subject_id"].nunique(),
                "g_true": g_true,
                "prior": pname,
                "n_labels": n_labels,

                "chain_cov": cov_counts["chain"] / NSIM,
                "chain_w": float(np.nanmean(widths["chain"])),

                "labeled_only_cov": cov_counts["labeled_only"] / NSIM,
                "labeled_only_w": float(np.nanmean(widths["labeled_only"])),

                "diff_cov": cov_counts["diff"] / NSIM,
                "diff_w": float(np.nanmean(widths["diff"])),

                "ppi_cov": cov_counts["ppi"] / NSIM,
                "ppi_w": float(np.nanmean(widths["ppi"])),

                "ppipp_cov": cov_counts["ppipp"] / NSIM,
                "ppipp_w": float(np.nanmean(widths["ppipp"])),
                "ppipp_lambda_mean": float(np.nanmean(ppipp_lambdas)),
                "ppipp_lambda_sd": float(np.nanstd(ppipp_lambdas, ddof=1)) if len(ppipp_lambdas) > 1 else 0.0,
            }

            all_rows.append(row)

    results = pd.DataFrame(all_rows)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    results.to_csv(output_csv, index=False)

    print("Saved:", output_csv)
    display(results)

    return results

In [ ]:
# ============================================================
# 5) Run full cohort
# ============================================================

results_full = run_coverage_experiment(
    full_df,
    dataset_name="full",
    input_csv_path=FULL_CSV,
    output_csv=OUTPUT_FULL_CSV,
    seed_offset=0,
)

In [ ]:
# ============================================================
# 6) Run age 65--70 subset
# ============================================================

results_6570 = run_coverage_experiment(
    age6570_df,
    dataset_name="age_65_70",
    input_csv_path=AGE6570_CSV,
    output_csv=OUTPUT_6570_CSV,
    seed_offset=500000,
)

In [ ]:
# ============================================================
# 7) Combine and save results
# ============================================================

combined_results = pd.concat([results_full, results_6570], ignore_index=True)
combined_results.to_csv(OUTPUT_COMBINED_CSV, index=False)

print("Saved combined results:", OUTPUT_COMBINED_CSV)
display(combined_results)

In [ ]:
# ============================================================
# 8) Paper-friendly summary tables
# ============================================================

def make_prior_summary_table(results, dataset_name, prior):
    columns = {
        "n_labels":"n", "chain_cov":"CRE_cov", "chain_w":"CRE_w",
        "labeled_only_cov":"LabeledOnly_cov", "labeled_only_w":"LabeledOnly_w",
        "diff_cov":"BinaryDiff_cov", "diff_w":"BinaryDiff_w",
        "ppi_cov":"PPI_cov", "ppi_w":"PPI_w",
        "ppipp_cov":"PPIpp_cov", "ppipp_w":"PPIpp_w",
    }
    sub=results.loc[results.dataset.eq(dataset_name)&results.prior.eq(prior)]
    return sub.sort_values("n_labels")[list(columns)].rename(columns=columns).reset_index(drop=True)

for cohort, suffix in [("full","full"),("age_65_70","6570")]:
    for prior in ["uniform","jeffreys"]:
        summary=make_prior_summary_table(combined_results,cohort,prior)
        summary.to_csv(OUTPUT_DIR/f"table_{suffix}_{prior}_summary.csv",index=False)
        print(cohort,prior)
        display(summary.round(3))


In [ ]:
# ============================================================
# 9) Plotting
# ============================================================

def plot_coverage_and_width(results: pd.DataFrame, dataset_name: str, output_dir: Path):
    """
    Save coverage and width plots for one dataset.

    PPI and PPI++ are prior-free, so they are averaged over duplicate prior rows.
    """
    sub_all = results[results["dataset"] == dataset_name].copy()
    title_name = DISPLAY_NAME.get(dataset_name, dataset_name)

    prior_free = (
        sub_all.groupby("n_labels", as_index=False)[["ppi_cov", "ppi_w", "ppipp_cov", "ppipp_w"]]
        .mean()
        .sort_values("n_labels")
    )

    # -------------------------
    # Coverage plot
    # -------------------------
    plt.figure(figsize=(10, 5))

    for prior in ["uniform", "jeffreys"]:
        subset = sub_all[sub_all["prior"] == prior].sort_values("n_labels")

        plt.plot(subset["n_labels"], subset["chain_cov"], "o-", label=f"CRE ({prior})")
        plt.plot(subset["n_labels"], subset["labeled_only_cov"], "s--", label=f"Labeled-only ({prior})")
        plt.plot(subset["n_labels"], subset["diff_cov"], "d-.", label=f"Binary Diff ({prior})")

    plt.plot(prior_free["n_labels"], prior_free["ppi_cov"], "^:", label="PPI")
    plt.plot(prior_free["n_labels"], prior_free["ppipp_cov"], "v:", label="PPI++")

    plt.axhline(0.95, linestyle="--", linewidth=1, label="Nominal 0.95")

    plt.ylim(0.3, 1.05)
    plt.xlabel("Number of labeled scans")
    plt.ylabel("Empirical coverage")
    plt.title(f"Coverage vs. Label Budget: {title_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    if SAVE_FIGURES:
        out_png = output_dir / f"fig_coverage_{dataset_name}_subject_oof_scan_age.png"
        out_pdf = output_dir / f"fig_coverage_{dataset_name}_subject_oof_scan_age.pdf"
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.savefig(out_pdf, bbox_inches="tight")
        print("Saved:", out_png)
        print("Saved:", out_pdf)

    plt.show()

    # -------------------------
    # Width plot
    # -------------------------
    plt.figure(figsize=(10, 5))

    for prior in ["uniform", "jeffreys"]:
        subset = sub_all[sub_all["prior"] == prior].sort_values("n_labels")

        plt.plot(subset["n_labels"], subset["chain_w"], "o-", label=f"CRE ({prior})")
        plt.plot(subset["n_labels"], subset["labeled_only_w"], "s--", label=f"Labeled-only ({prior})")
        plt.plot(subset["n_labels"], subset["diff_w"], "d-.", label=f"Binary Diff ({prior})")

    plt.plot(prior_free["n_labels"], prior_free["ppi_w"], "^:", label="PPI")
    plt.plot(prior_free["n_labels"], prior_free["ppipp_w"], "v:", label="PPI++")

    plt.xlabel("Number of labeled scans")
    plt.ylabel("Mean 95% interval width")
    plt.title(f"Interval Width vs. Label Budget: {title_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    if SAVE_FIGURES:
        out_png = output_dir / f"fig_width_{dataset_name}_subject_oof_scan_age.png"
        out_pdf = output_dir / f"fig_width_{dataset_name}_subject_oof_scan_age.pdf"
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.savefig(out_pdf, bbox_inches="tight")
        print("Saved:", out_png)
        print("Saved:", out_pdf)

    plt.show()


plot_coverage_and_width(combined_results, "full", OUTPUT_DIR)
plot_coverage_and_width(combined_results, "age_65_70", OUTPUT_DIR)